In [0]:
stg_posts_df = spark.table('stackexchange_dataset.default.stg_posts')


In [0]:
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def posts_top_tags(df: DataFrame) -> DataFrame:
    return(
        df
        .withColumn('ExplodedTags', F.explode('TagsArray'))
        .groupBy(F.col('ExplodedTags'))
        .agg(F.approx_count_distinct(F.col('PostId')).alias('TagsCount'))
        .orderBy(F.col('TagsCount').desc())
    )

marts_top_tags_df = posts_top_tags(stg_posts_df)
(
    marts_top_tags_df
    .write
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('stackexchange_dataset.default.marts_top_tags')
)
